#1. Instalación de librerías y configuración inicial

In [ ]:
!sudo apt-get install tesseract-ocr > /dev/null
!sudo apt-get install libtesseract-dev > /dev/null
!pip install -q pytesseract easyocr opencv-python-headless matplotlib tqdm
!pip install -q paddlepaddle
!pip install -q paddleocr
!pip install -q langchain==0.0.350
!pip install -q langchain-community==0.0.3
!pip install -q jellyfish
!pip install -q ultralytics

debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 3.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.6/300.6 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.7/193.7 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 6.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 4.3 MB/s eta 0

In [ ]:
!pip install --upgrade scipy scikit-learn pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 91.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 82.7 MB/s eta 0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalled scipy-1.16.3
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is t

In [ ]:
import glob
import os
import random
import re
import tarfile
import time
import yaml

import cv2
from huggingface_hub import hf_hub_download
import jellyfish
import math
import matplotlib.pyplot as plt
import numpy as np
import paddle
import pandas as pd
from sklearn.cluster import DBSCAN
import torch
import torchvision.transforms as T
from ultralytics.utils.nms import TorchNMS
import warnings

import easyocr
import pytesseract
from paddleocr import PaddleOCR

/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.


In [ ]:
def set_global_seed(seed: int = 42):
    """
    Fija todas las semillas necesarias para asegurar reproducibilidad
    en experimentos.
    """
    # Python y entorno
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)

    # NumPy
    np.random.seed(seed)

    # PyTorch
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # CuDNN (para asegurar resultados deterministas en GPU)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    print(f"Semillas fijadas en {seed}")

In [ ]:
seed = 42
set_global_seed(seed)

Semillas fijadas en 42


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#2. Carga de subconjunto de prueba

In [ ]:
# Preparación del conjunto de datos
repo_id = "U20212419/digits"
extra_tar = hf_hub_download(repo_id=repo_id,
                            filename="extra/data.tar",
                            repo_type="dataset")
extra_yaml = hf_hub_download(repo_id=repo_id,
                             filename="extra/data.yaml",
                             repo_type="dataset")

if not os.path.exists("./data"):
    with tarfile.open(extra_tar, "r") as tar:
        tar.extractall(".", filter="data")

with open(extra_yaml, "r") as f:
    class_names = yaml.safe_load(f)['names']

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


extra/data.tar:   0%|          | 0.00/2.80M [00:00<?, ?B/s]

data.yaml:   0%|          | 0.00/130 [00:00<?, ?B/s]

In [ ]:
# Se usará la columna question_amount
repo_id_videos = "U20212419/videos"

# Descargar metadata.csv
metadata_csv_path = hf_hub_download(repo_id=repo_id_videos,
                                    filename="metadata.csv",
                                    repo_type="dataset")

print("Archivo descargado:")
print("metadata.csv:", metadata_csv_path)

# Cargar el archivo a un DataFrame
df_videos = pd.read_csv(metadata_csv_path)

print("\nCSV cargado exitosamente. Primeras 5 filas:")
print(df_videos.head())
df_videos['video_key'] = df_videos['name'].str.split('.').str[0]
qa_mapping = pd.Series(df_videos['question_amount'].values,
                        index=df_videos['video_key']).to_dict()

metadata.csv: 0.00B [00:00, ?B/s]

Archivo descargado:
metadata.csv: /root/.cache/huggingface/hub/datasets--U20212419--videos/snapshots/7c4f861eaabee8c23b67a66b3eb446975a61326a/metadata.csv

CSV cargado exitosamente. Primeras 5 filas:
            name                   format  size_bytes  length_seconds  width  \
0  video_001.mp4  mov,mp4,m4a,3gp,3g2,mj2  2609345289          144.92   3840   
1  video_002.mp4  mov,mp4,m4a,3gp,3g2,mj2  1085200789          165.41   1920   
2  video_003.mp4  mov,mp4,m4a,3gp,3g2,mj2  1122202727          170.98   1920   
3  video_004.mp4  mov,mp4,m4a,3gp,3g2,mj2   793332274          120.93   1920   
4  video_005.mp4  mov,mp4,m4a,3gp,3g2,mj2  1145469583          174.36   1920   

   height  avg_frame_rate  booklet_count  question_amount  
0    2160           60.03             57                4  
1    1080           60.02             58                4  
2    1080           60.01             59                4  
3    1080           60.02             53                4  
4    1080          

In [ ]:
def convert_obb_to_yolo(src_folder, dst_folder):
    """
    Convierte las labels del formato OBB
    <class> (x1 y1 x2 y2 x3 y3 x4 y4)
    al formato YOLO para detección
    <class> <x_center> <y_center> <width> <height>
    """
    os.makedirs(dst_folder, exist_ok=True)
    label_files = glob.glob(os.path.join(src_folder, "*.txt"))

    for label_path in label_files:
        with open(label_path, "r") as f:
            lines = f.readlines()

        new_lines = []
        for line in lines:
            parts = line.strip().split()
            if len(parts) < 9:
                continue

            # Clase
            cls_id = parts[0]
            # x1 y1 x2 y2 x3 y3 x4 y4
            coords = list(map(float, parts[1:9]))
            xs = coords[0::2]
            ys = coords[1::2]

            # Bounding box (min/max)
            x_min, x_max = min(xs), max(xs)
            y_min, y_max = min(ys), max(ys)

            # Convertir a formato YOLO
            # <class> <x_center> <y_center> <width> <height>
            x_center = (x_min + x_max) / 2
            y_center = (y_min + y_max) / 2
            width = x_max - x_min
            height = y_max - y_min

            new_line = (f"{cls_id} {x_center:.6f} "
                        f"{y_center:.6f} {width:.6f} {height:.6f}\n")
            new_lines.append(new_line)

        # Guardar nueva etiqueta
        dst_path = os.path.join(dst_folder, os.path.basename(label_path))
        with open(dst_path, "w") as f:
            f.writelines(new_lines)

    print(f"Conversion completa: {len(label_files)} archivos procesados.")

In [ ]:
# Convertir las labels a formato YOLO para detección
src_folder = "data/labels/extra"
dst_folder = "data/labels/extra_new"
convert_obb_to_yolo(src_folder, dst_folder)

Conversion completa: 154 archivos procesados.


In [ ]:
!mv data/labels/extra data/labels/original
!mv data/labels/extra_new data/labels/extra

#3. Pruebas comparativas

In [ ]:
warnings.filterwarnings("ignore", message=".*pin_memory.*")
paddle.set_flags({'FLAGS_enable_pir_api': 0})

# Carga de modelos (OCR y propio)
print("\nCargando modelos...")

# OCR existentes
# PyTesseract no necesita ser cargado
reader_easy = easyocr.Reader(['es', 'en'], gpu=False)
paddle_ocr = PaddleOCR(use_textline_orientation=False, lang='en',
                       enable_mkldnn=False)
print("Modelos existentes de OCR cargados.")

# Modelo propio (YOLO + ResNet)
device = torch.device('cpu')
try:
    yolo_model = torch.jit.load("digits_yolo_cpu.pt",
                                map_location=device).eval()
    resnet_model = torch.jit.load("digits_resnet.pt",
                                  map_location=device).eval()
    print("Modelos Propios cargados.")
except:
    print("ADVERTENCIA: No se encontraron 'digits_yolo_cpu.pt' o "
          "'digits_resnet.pt'. La columna 'CUSTOM' fallará.")
    yolo_model = None


Cargando modelos...


Creating model: ('PP-LCNet_x1_0_doc_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-LCNet_x1_0_doc_ori`.
Creating model: ('UVDoc', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/UVDoc`.
Creating model: ('PP-OCRv5_server_det', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-OCRv5_server_det`.
Creating model: ('en_PP-OCRv5_mobile_rec', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/en_PP-OCRv5_mobile_rec`.


Modelos existentes de OCR cargados.
Modelos Propios cargados.


## Funciones de procesamiento e inferencia

In [ ]:
def _find_nearest_digit_slot(decimal_char, main_digits, y_min_anchor,
                             slot_height_real, num_rows=9):
    """
    Encuentra la casilla más cercana a un dígito,
    en función de sus coordendas centrales.
    """
    if not main_digits: return -1
    min_dist = float('inf')
    nearest_digit_y_center = -1
    dec_x_center = (decimal_char['x1'] + decimal_char['x2']) / 2
    dec_y_center = decimal_char['y_center']

    for digit in main_digits:
        dig_x_center = (digit['x1'] + digit['x2']) / 2
        dig_y_center = digit['y_center']
        dist = (math.sqrt((dec_x_center - dig_x_center)**2 +
                (dec_y_center - dig_y_center)**2))
        if dist < min_dist:
            min_dist = dist
            nearest_digit_y_center = dig_y_center

    if slot_height_real == 0: slot_index = 0
    else:
        relative_y = nearest_digit_y_center - y_min_anchor
        slot_index = int(round(relative_y / slot_height_real))
    return max(0, min(slot_index, num_rows - 1))

In [ ]:
def sanitize_score_string(s: str) -> str:
    """Limpia la cadena para que represente un puntaje decimal válido."""
    if not s: return ""
    s_out = ""
    dot_found = False
    for char in s:
        if char == '.':
            if not dot_found:
                s_out += char
                dot_found = True
        elif char.isdigit():
            s_out += char
    return s_out.strip('.')

In [ ]:
def finalize_scores_by_slotting(char_detections: list,
                                question_amount: int,
                                num_rows: int = 9) -> dict:
    """
    Agrupa los puntajes colocando los caracteres en cada casilla según
    su centro vertical y_center.
    """
    # Separa los dígitos y los separadores decimales
    main_digits = []
    satellite_chars = []
    for char in char_detections:
        if char['char'].isdigit():
            main_digits.append(char)
        elif char['char'] == '.':
            satellite_chars.append(char)

    # No se detectaron dígitos
    if not main_digits:
        output_json = {}
        for i in range(num_rows - 1):
            output_json[f"question_{i+1}"] = 0.0
        output_json["total_score"] = 0.0
        return output_json

    # Haya los centros verticales utilizando DBSCAN
    y_centers_digits = []
    char_heights = []
    for char in main_digits:
        y_centers_digits.append(char['y_center'])
        height = char['y2'] - char['y1']
        if height > 0:
            char_heights.append(height)

    y_min_anchor = min(y_centers_digits)
    y_max_anchor = max(y_centers_digits)

    if char_heights:
        avg_char_height = np.mean(char_heights)
        # 75% de la altura promedio de un dígito
        eps = avg_char_height * 0.75

        y_centers_np = np.array(y_centers_digits).reshape(-1, 1)
        db = DBSCAN(eps=eps, min_samples=1).fit(y_centers_np)
        labels = db.labels_

        stable_row_centers = []
        for label in set(labels):
            if label == -1: continue
            points_in_cluster = y_centers_np[labels == label]
            stable_row_centers.append(np.mean(points_in_cluster))

        if stable_row_centers:
            y_min_anchor = min(stable_row_centers)
            y_max_anchor = max(stable_row_centers)

    # Calcula la altura de la tabla y de una casilla
    table_height_real = y_max_anchor - y_min_anchor

    if table_height_real == 0 or (num_rows - 1) == 0:
        slot_height_real = 0
    else:
        slot_height_real = table_height_real / (num_rows - 1)

    # Asigna caracteres a casillas
    slots = [[] for _ in range(num_rows)]

    # Asigna los dígitos
    for char in main_digits:
        if slot_height_real == 0:
            slot_index = 0
        else:
            relative_y = char['y_center'] - y_min_anchor
            slot_index = int(round(relative_y / slot_height_real))

        slot_index = max(0, min(slot_index, num_rows - 1))
        slots[slot_index].append(char)

    # Asigna los separadores decimales
    for char in satellite_chars:
        slot_index = _find_nearest_digit_slot(
            char, main_digits, y_min_anchor, slot_height_real, num_rows
        )
        if slot_index != -1:
            slots[slot_index].append(char)

    # Agrupa los puntajes por cada casilla
    final_scores = {}
    for i in range(num_rows):
        chars_in_this_row = slots[i]
        score_value = 0.0

        if chars_in_this_row:
            for d in chars_in_this_row:
                d['x_center'] = (d['x1'] + d['x2']) / 2
            sorted_chars = sorted(chars_in_this_row,
                                  key=lambda d: d['x_center'])
            score_str = "".join([d['char'] for d in sorted_chars])
            sanitized_str = sanitize_score_string(score_str)

            if sanitized_str:
                try:
                    score_value = float(sanitized_str)
                except ValueError:
                    score_value = 0.0

        # Manejo de valores absurdos: solo el puntaje total puede ser 20
        if score_value >= 20.0 and i < num_rows - 1:
            score_value = (score_value /
                           pow(10, len(str(int(score_value))) - 1))
        if score_value > 20.0 and i == num_rows - 1:
            score_value = (score_value /
                           pow(10, len(str(int(score_value))) - 1))
        final_scores[i] = score_value

    output_json = {}
    for i in range(num_rows - 1):
        key_name = f"question_{i+1}"
        if i >= question_amount:
            # No se coloca un valor para preguntas no existentes
            continue
        else:
            output_json[key_name] = final_scores[i]
    output_json["total_score"] = final_scores[num_rows - 1]

    return output_json

In [ ]:
def get_ground_truth_text(label_path, img_w, img_h, question_amount):
    """Construye la cadena de puntajes reales usando las etiquetas."""
    if not os.path.exists(label_path): return ""
    detections = []
    with open(label_path, 'r') as f:
        for line in f.readlines():
            c, x, y, w, h = map(float, line.split())
            # Convertir a píxel para simular entrada de YOLO
            x1 = int((x - w/2) * img_w)
            y1 = int((y - h/2) * img_h)
            x2 = int((x + w/2) * img_w)
            y2 = int((y + h/2) * img_h)
            detections.append({
                'char': '.' if c == 10 else str(int(c)),
                'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2,
                'y_center': (y1 + y2) / 2
            })

    scores_map = finalize_scores_by_slotting(detections, question_amount)
    # Convertir floats a string para comparación
    return " ".join([str(v) for k, v in
                     scores_map.items() if v is not None])

In [ ]:
def limpiar_output_ocr(text):
    """
    Normaliza el texto para hacerlo comparable con el Ground Truth:
    1. Reemplaza saltos de línea por espacios.
    2. Reemplaza comas por puntos (14,5 -> 14.5).
    3. Elimina todo carácter que no sea dígito, punto o espacio.
    4. Normaliza espacios.
    """
    if not text: return ""
    text = text.replace('\n', ' ').replace(',', '.')
    text = re.sub(r'[^0-9. ]', '', text)
    return re.sub(r'\s+', ' ', text).strip()

In [ ]:
def scale_coords(coords, img_original_shape, ratio, pad):
    """Escala las coordenadas a la proporción de la imagen original."""
    pad_x, pad_y = pad
    coords[:, [0, 2]] -= pad_x
    coords[:, [1, 3]] -= pad_y
    coords[:, :4] /= ratio
    coords[:, 0].clamp_(0, img_original_shape[1])
    coords[:, 1].clamp_(0, img_original_shape[0])
    coords[:, 2].clamp_(0, img_original_shape[1])
    coords[:, 3].clamp_(0, img_original_shape[0])
    return coords

In [ ]:
def letterbox(img, new_shape=1280, color=(114, 114, 114)):
    """Redimensiona las imágenes y agrega padding de un color uniforme."""
    shape = img.shape[:2]
    if isinstance(new_shape, int): new_shape = (new_shape, new_shape)
    r = min(new_shape[0] / shape[0], new_shape[1] / shape[1])
    new_unpad = int(round(shape[1] * r)), int(round(shape[0] * r))
    dw, dh = new_shape[1] - new_unpad[0], new_shape[0] - new_unpad[1]
    dw, dh = dw / 2, dh / 2
    if shape[::-1] != new_unpad:
        img = cv2.resize(img, new_unpad, interpolation=cv2.INTER_LINEAR)
    top, bottom = int(round(dh - 0.1)), int(round(dh + 0.1))
    left, right = int(round(dw - 0.1)), int(round(dw + 0.1))
    img = cv2.copyMakeBorder(img, top, bottom, left, right,
                             cv2.BORDER_CONSTANT, value=color)
    return img, r, (left, top)

In [ ]:
def filter_outliers(detections, img_orig, ratio, pad, device="cpu",
                    threshold_ratio=0.8, num_rows=9):
    """
    Descarta las bounding boxes que están fuera de una columa vertical
    definida por un umbral definido empíricamente.
    """
    # Escala las coordenadas a las originales
    detections[:, :4] = scale_coords(detections[:, :4],
                                     img_orig.shape, ratio, pad)

    img_w = img_orig.shape[1]
    # Define un umbral basado en el ancho de la imagen
    x_threshold = img_w * threshold_ratio

    img_h = img_orig.shape[0]
    y_threshold = img_h // (num_rows + 1)

    # Filtra las detecciones basándose en la coordenada x1
    good_detections = []
    for det in detections:
        # Usa el borde izquierdo x1 para ser más certero que con x_center
        if det[0] < x_threshold and det[1] > y_threshold:
            good_detections.append(det)

    if not good_detections:
        # Si no quedan bounding boxes, se crea un tensor vacío
        detections = torch.empty((0, 6), device=device)
    else:
        # 'detections' ahora solo contiene las bounding boxes aceptadas
        detections = torch.stack(good_detections)

    return detections

In [ ]:
def predict_custom_pipeline(img_orig, question_amount=8):
    """Predicciones utilizando el pipeline propio."""
    if yolo_model is None: return ""

    img_rgb = cv2.cvtColor(img_orig, cv2.COLOR_BGR2RGB)

    crop_resized, ratio, pad = letterbox(img_rgb, new_shape=1280)
    crop_tensor = torch.from_numpy(
        crop_resized
    ).permute(2, 0, 1).float()/255.0
    crop_tensor = crop_tensor.unsqueeze(0).to(device)

    with torch.no_grad():
        preds = yolo_model(crop_tensor)[0]

    boxes = preds[:, :4]
    scores = preds[:, 4]

    # NMS
    keep = TorchNMS.nms(boxes, scores, iou_threshold=0.5)
    detections = preds[keep]

    detections = filter_outliers(detections, img_orig, ratio,
                                 pad, device, threshold_ratio=0.8)

    if len(detections) == 0: return ""

    # Clasificación
    # El modelo fue entrenado con la clase 10 en el índice 2
    index_to_class = {
        0: '0',
        1: '1',
        2: '.',
        3: '2',
        4: '3',
        5: '4',
        6: '5',
        7: '6',
        8: '7',
        9: '8',
        10: '9'
    }
    detections_for_assembly = []

    resnet_mean = [0.7138324946621879,
                   0.6752742936984362,
                   0.6944984441793525]
    resnet_std = [0.06064024192479617,
                  0.08277346212477447,
                  0.07542827455486965]
    preprocess_resnet = T.Compose([
        T.ToPILImage(),
        T.Resize((224, 224)),
        T.ToTensor(),
        T.Normalize(resnet_mean, resnet_std)
    ])

    for det in detections:
        x1f, y1f, x2f, y2f, conf, cls = det.cpu().numpy()

        # Asegurarse de que las coordenadas estén dentro de los límites
        h, w = img_orig.shape[:2]
        x1 = max(0, int(np.floor(x1f)))
        y1 = max(0, int(np.floor(y1f)))
        x2 = min(w, int(np.ceil(x2f)))
        y2 = min(h, int(np.ceil(y2f)))

        # Evita recortes vacíos
        if x1 >= x2 or y1 >= y2:
            continue

        digit_crop = img_orig[y1:y2, x1:x2]
        if digit_crop.size == 0:
            continue  # Salta recortes inválidos

        digit_crop_rgb = cv2.cvtColor(digit_crop, cv2.COLOR_BGR2RGB)
        digit_tensor = preprocess_resnet(
            digit_crop_rgb
        ).unsqueeze(0).to(device)

        with torch.no_grad():
            digit_pred = resnet_model(digit_tensor)
            digit_class_idx = digit_pred.argmax(dim=1).item()
            if digit_class_idx not in index_to_class:
                continue  # Salta clases desconocidas
            digit_class_name = index_to_class[digit_class_idx]

        detections_for_assembly.append({
            'char': digit_class_name,
            'x1': x1, 'x2': x2,
            'y_center': (y1 + y2)/2,
            'y1': y1, 'y2': y2
        })

    # Ensamblaje (clustering)
    scores_map = finalize_scores_by_slotting(detections_for_assembly,
                                             question_amount)

    # Convertir diccionario a string plano para comparar con el GT
    # Ej: {'q_0': '14.5', 'q_1': '10'} -> "14.5 10"
    return " ".join([str(v) for k, v in
                     scores_map.items() if v is not None])

## Mediciones finales

In [ ]:
# Usar el subconjunto de prueba
test_subset = (glob.glob("data/images/extra/*.jpg") +
              glob.glob("data/images/extra/*.png"))

# Encabezado de la tabla
print(f"{'IMAGEN':<35} | {'GROUND TRUTH (REAL)':<50} | "
      f"{'PYTESSERACT':<50} | {'EASYOCR':<50} | "
      f"{'PADDLEOCR':<50} | {'CUSTOM PIPELINE':<50}")
print("-" * 285)

metrics = {'tess': [], 'easy': [], 'paddle': [], 'custom': []}
times   = {'tess': [], 'easy': [], 'paddle': [], 'custom': []}

# Mediciones sobre el subconjunto de prueba
for img_path in test_subset:
    img = cv2.imread(img_path)
    if img is None: continue

    lbl_path = (img_path
                .replace("images", "labels")
                .replace(".jpg", ".txt")
                .replace(".png", ".txt"))

    pattern = r"video_[0-9]{3}"
    matches = re.findall(pattern, img_path)
    question_amount = qa_mapping.get(matches[0] if matches else '')

    # Ground truth
    real_text = get_ground_truth_text(lbl_path, img.shape[1],
                                      img.shape[0], question_amount)

    # Tesseract
    start = time.time()
    tess_raw = pytesseract.image_to_string(img, config='--psm 6')
    end = time.time()
    times['tess'].append(end - start)
    tess = limpiar_output_ocr(tess_raw)

    # EasyOCR
    start = time.time()
    easy_raw = reader_easy.readtext(img, detail=0)
    end = time.time()
    times['easy'].append(end - start)
    easy = limpiar_output_ocr(" ".join(easy_raw))

    # PaddleOCR
    start = time.time()
    try:
        # Ajuste para formato de diccionario de Paddle
        pred_p = paddle_ocr.predict(img)
        if (isinstance(pred_p, list) and len(pred_p) > 0 and
            isinstance(pred_p[0], dict)):
             padd_list = pred_p[0].get('rec_texts', [])
        else:
             padd_list = []  # Fallback
        padd_raw = " ".join(padd_list)
    except Exception as e:
        print(e)
        padd_raw = ""
    end = time.time()
    times['paddle'].append(end - start)
    padd = limpiar_output_ocr(padd_raw)

    # Custom pipeline
    start = time.time()
    custom = predict_custom_pipeline(img, question_amount)
    end = time.time()
    times['custom'].append(end - start)

    # Puntajes
    metrics['tess'].append(
        jellyfish.jaro_winkler_similarity(real_text, tess)
    )
    metrics['easy'].append(
        jellyfish.jaro_winkler_similarity(real_text, easy)
    )
    metrics['paddle'].append(
        jellyfish.jaro_winkler_similarity(real_text, padd)
    )
    metrics['custom'].append(
        jellyfish.jaro_winkler_similarity(real_text, custom)
    )

    # Imprimir cadenas
    # Se usa una cadena vacía "---" si el resultado es nulo
    r_ver = real_text[:50] if real_text else "---"
    t_ver = tess[:50] if tess else "---"
    e_ver = easy[:50] if easy else "---"
    p_ver = padd[:50] if padd else "---"
    c_ver = custom[:50] if custom else "---"

    print(f"{os.path.basename(img_path)[:35]:<35} | {r_ver:<50} | "
          f"{t_ver:<50} | {e_ver:<50} | {p_ver:<50} | {c_ver:<50}")

print("-" * 285)
print("RESULTADOS FINALES")
print("="*80)
print(f"{'MODELO':<15} | "
      f"{'AVG. JARO-WINKLER':<20} | "
      f"{'TIEMPO PROMEDIO (s)':<20}")
print("-" * 80)
print(f"Tesseract:      | {np.mean(metrics['tess']):.2%}"
      f"               | {np.mean(times['tess']):.4f} s")
print(f"EasyOCR:        | {np.mean(metrics['easy']):.2%}"
      f"               | {np.mean(times['easy']):.4f} s")
print(f"PaddleOCR:      | {np.mean(metrics['paddle']):.2%}"
      f"               | {np.mean(times['paddle']):.4f} s")
print(f"TU PIPELINE:    | {np.mean(metrics['custom']):.2%}"
      f"               | {np.mean(times['custom']):.4f} s")
print("="*80)

IMAGEN                              | GROUND TRUTH (REAL)                                | PYTESSERACT                                        | EASYOCR                                            | PADDLEOCR                                          | CUSTOM PIPELINE                                   
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
video_006_frame_00052.jpg           | 5.0 5.0 3.5 5.0 18.5                               | ---                                                | 5.0 510 3.5 5.0 0 18.50                            | 5.0 5.0 3.5 5.0 18.50                              | 5.0 5.0 3.5 5.0 18.5                              
video_006_frame_00011.jpg           | 4.0 5.0 4.75 4.75 18.5                             | ---                  